# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we'll print out all available Record Sets in the dataset, referencing them by their `@id` as per the Croissant specification.

> **Note:** Entities are always referenced by their `@id`.


In [ ]:
# List all available Record Sets (by @id and name)
record_sets = list(metadata.record_sets)
print("Record Sets (@id and name):")
for rs in record_sets:
    print(f"  @id: {rs['@id']}  | name: {rs.get('name', 'N/A')}")

In [ ]:
# For each record set, print out its fields (by @id and name)
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        # The field may be an @id reference or dict
        if isinstance(field, str):
            # Try to resolve
            field_obj = next((f for f in metadata.fields if f['@id'] == field), None)
        elif isinstance(field, dict):
            field_obj = field
        else:
            field_obj = None
        if field_obj:
            print(f"    @id: {field_obj['@id']}  | name: {field_obj.get('name', 'N/A')}")
        else:
            print(f"    @id: {field}  | name: not found")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract all record sets and store them as pandas DataFrames for analysis. Make sure to use the `@id` fields.

In [ ]:
# Extract data from each record set
# Build a list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for Record Set @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} rows. Columns:", df.columns.tolist())
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading: {e}")

List the fields for a particular record set (`@id`). If you know which record set you want to analyze, replace the variable below with that `@id`. We'll also preview the first few rows.

In [ ]:
# Choose a record set to analyze (update this @id from the overview step above)
main_record_set_id = record_set_ids[0]  # update if you want a different record set
if main_record_set_id in dataframes:
    print("Columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No DataFrame loaded for @id: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, grouping, and handling outliers.
We'll select a field (`@id`, which is the column name, as in previous step) for numeric analysis.

> **Remember to use only `@id`-based column names!**

In [ ]:
# Example: Pick a numeric field (replace below)
df = dataframes[main_record_set_id].copy()

# Try to pick a numeric column
possible_numeric = df.select_dtypes(include='number').columns.tolist()
if possible_numeric:
    numeric_field = possible_numeric[0]  # or set as needed
else:
    # Try to detect numeric even if type is object
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    possible_numeric = df.select_dtypes(include='number').columns.tolist()
    if possible_numeric:
        numeric_field = possible_numeric[0]
    else:
        raise ValueError("No numeric field found in the record set!")

print(f"Using numeric field for analysis: {numeric_field}")

# Filter out records where the numeric_field > threshold
threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field if available
cat_fields = [col for col in df.columns if df[col].nunique() < 8 and col != numeric_field]
if cat_fields:
    group_field = cat_fields[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    display(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If a group field was found, plot a boxplot
if 'group_field' in locals():
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded a mass spectrometry proteomics dataset defined by a Croissant schema using mlcroissant
- Explored the dataset's metadata, structure, and fields referencing all elements by their Croissant `@id`
- Extracted and previewed records for available record sets
- Applied basic filtering, normalization, and grouped summaries (EDA)
- Visualized numeric data distributions and group differences

This workflow demonstrates how `mlcroissant` enables FAIR, schema-based data exploration leveraging machine-readable standards. For deeper analysis, refine field selection and apply advanced domain-specific methods!